# 運用② — ドリフト検知と再学習の発火条件

出典：第06章 03節／パス：`06_ops/02_improved.ipynb`（改善モデル）／パッケージ：numpy・pandas・scikit-learn／実行時間の目安：約1分

このノート単体で実行できる。年収分布がずれた想定で、入力分布（PSI、早期警報）と帯別貸倒率（貸倒の確定後、学習に使っていないholdoutを基準）を測り、再学習条件の発火を試す。
ずれ幅・閾値は仮置きであり、実務の根拠ではない。

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RS = 0


def make_ops_data(n: int, income_shift: float = 0.0, seed: int = 61) -> pd.DataFrame:
    r = np.random.default_rng(seed)
    income = np.clip(r.normal(450.0 + income_shift, 150.0, size=n), 150.0, 1500.0)
    past_due = np.clip(r.poisson(0.4, size=n), 0, 5)
    use_rate = r.uniform(0.0, 1.0, size=n)
    housing = r.choice(["rent", "own", "company"], size=n, p=[0.55, 0.35, 0.10])
    logit = -3.4 + 0.9 * past_due + 1.2 * use_rate - 0.003 * (income - 450.0) + 0.2 * (housing == "rent")
    y = (r.uniform(size=n) < 1.0 / (1.0 + np.exp(-logit))).astype(int)
    return pd.DataFrame({"income": income, "past_due": past_due,
                         "use_rate": use_rate, "housing": housing, "default": y})


def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    qs = np.quantile(expected, np.linspace(0, 1, bins + 1))
    qs[0], qs[-1] = -np.inf, np.inf
    e, _ = np.histogram(expected, bins=qs)
    a, _ = np.histogram(actual, bins=qs)
    e = e / e.sum()
    a = a / a.sum()
    e = np.clip(e, 1e-6, None)
    a = np.clip(a, 1e-6, None)
    return float(((a - e) * np.log(a / e)).sum())


try:
    base = pd.read_csv("input/ops_base.csv")
    print("Inputを使用: input/ops_base.csv")
except FileNotFoundError:
    base = make_ops_data(6000, income_shift=0.0)
    print("Inputが無いため合成データを生成")
try:
    new = pd.read_csv("input/ops_new.csv")
    print("Inputを使用: input/ops_new.csv")
except FileNotFoundError:
    # 景気悪化で年収が下がった想定
    new = make_ops_data(2000, income_shift=-120.0, seed=62)
    print("Inputが無いため合成データを生成")
print(f"学習時 平均年収: {base['income'].mean():.1f} / 新着 平均年収: {new['income'].mean():.1f}")
print(f"年収のPSI={psi(base['income'].to_numpy(), new['income'].to_numpy()):.3f}（目安: 0.1未満小・0.25超大）")

Inputを使用: input/ops_base.csv
Inputを使用: input/ops_new.csv
学習時 平均年収: 452.9 / 新着 平均年収: 337.0
年収のPSI=0.647（目安: 0.1未満小・0.25超大）


In [2]:
Xb, yb = base.drop(columns=["default"]), base["default"]
# 学習に使わなかった25%（holdout）を「学習時」の基準にする。学習に使った行で測ると当てはまり過ぎで甘く出る
X_tr, X_ho, y_tr, y_ho = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=RS)
pre = ColumnTransformer([
    ("num", StandardScaler(), ["income", "past_due", "use_rate"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["housing"]),
])
pipe = make_pipeline(pre, HistGradientBoostingClassifier(max_iter=100, random_state=RS)).fit(X_tr, y_tr)
ref = X_ho.assign(default=y_ho.to_numpy())
for name, d in [("学習時(holdout)", ref), ("新着", new)]:
    p = pipe.predict_proba(d.drop(columns=["default"]))[:, 1]
    band = np.where(p >= 0.30, "deny", np.where(p <= 0.05, "approve", "manual"))
    out = []
    for b in ["approve", "manual", "deny"]:
        m = band == b
        out.append(f"{b}:{m.mean():.3f}(貸倒率{d['default'].to_numpy()[m].mean():.3f})")
    print(f"[{name}] " + " / ".join(out))

[学習時(holdout)] approve:0.449(貸倒率0.047) / manual:0.487(貸倒率0.116) / deny:0.064(貸倒率0.365)
[新着] approve:0.333(貸倒率0.095) / manual:0.555(貸倒率0.167) / deny:0.112(貸倒率0.359)


In [3]:
# 監視は二段にする。①PSI：申込が来た時点で測れる早期警報 ②帯別貸倒率：貸倒の確定（数か月〜1年）を待って測る発火条件
psi_income = psi(base["income"].to_numpy(), new["income"].to_numpy())
print(f"[早期警報] 年収のPSI={psi_income:.3f} → " + ("警報（貸倒の確定を待つ間、自動承認帯を絞って監視を強める）" if psi_income > 0.25 else "平常"))
p_ref = pipe.predict_proba(ref.drop(columns=["default"]))[:, 1]
p_new = pipe.predict_proba(new.drop(columns=["default"]))[:, 1]
deny_ref = ref["default"].to_numpy()[p_ref >= 0.30].mean()
deny_new = new["default"].to_numpy()[p_new >= 0.30].mean()
appr_ref = ref["default"].to_numpy()[p_ref <= 0.05].mean()
appr_new = new["default"].to_numpy()[p_new <= 0.05].mean()
print(f"deny帯の貸倒率: 学習時 {deny_ref:.3f} → 新着 {deny_new:.3f}（差 {deny_new - deny_ref:+.3f}）")
print(f"approve帯の貸倒率: 学習時 {appr_ref:.3f} → 新着 {appr_new:.3f}（差 {appr_new - appr_ref:+.3f}）")
# 再学習の発火条件（仮置き）：貸倒の確定後、自動承認帯の貸倒率が学習時（holdout）から3pt以上悪化したら発火
fire = appr_new - appr_ref >= 0.03
print("[確定後の発火] 再学習: " + ("発火（条件を満たした）" if fire else "待機（条件未達）"))
assert np.isfinite(p_new).all()

[早期警報] 年収のPSI=0.647 → 警報（貸倒の確定を待つ間、自動承認帯を絞って監視を強める）
deny帯の貸倒率: 学習時 0.365 → 新着 0.359（差 -0.006）
approve帯の貸倒率: 学習時 0.047 → 新着 0.095（差 +0.047）
[確定後の発火] 再学習: 発火（条件を満たした）


In [4]:
# 改善結果：監視の要点をテスト相当の新着データで1回だけ測る
print(f"[改善結果] 年収のPSI={psi_income:.3f}（申込時点で分布変化を検出）")
print(f"[改善結果] approve帯の貸倒率 {appr_ref:.3f} → {appr_new:.3f}（約{appr_new / appr_ref:.1f}倍。貸倒の確定後に測定）")
print("[改善結果] 再学習: " + ("発火（条件を満たした）" if fire else "待機（条件未達）"))

[改善結果] 年収のPSI=0.647（申込時点で分布変化を検出）
[改善結果] approve帯の貸倒率 0.047 → 0.095（約2.0倍。貸倒の確定後に測定）
[改善結果] 再学習: 発火（条件を満たした）


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
plt.figure()
plt.hist(base["income"].to_numpy(), bins=30, alpha=0.6, label="train-time")
plt.hist(new["income"].to_numpy(), bins=30, alpha=0.6, label="new")
plt.xlabel("income"); plt.ylabel("count"); plt.legend(); plt.title("income shift")
plt.savefig(f"{FIG}/code-result-j06-01.png", dpi=150, bbox_inches="tight"); plt.close()
p_b = pipe.predict_proba(ref.drop(columns=["default"]))[:, 1]
p_n = pipe.predict_proba(new.drop(columns=["default"]))[:, 1]
def bands(p, d):
    b = np.where(p >= 0.30, "deny", np.where(p <= 0.05, "approve", "manual"))
    return [d[b == k].mean() for k in ["approve", "manual", "deny"]]
x = np.arange(3); w = 0.35
plt.figure()
plt.bar(x - w / 2, bands(p_b, ref["default"].to_numpy()), w, label="train-time (holdout)")
plt.bar(x + w / 2, bands(p_n, new["default"].to_numpy()), w, label="new")
plt.xticks(x, ["approve", "manual", "deny"]); plt.ylabel("default rate"); plt.legend()
plt.title("band default rates")
plt.savefig(f"{FIG}/code-result-j06-02.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j06-01.png, code-result-j06-02.png")

saved code-result-j06-01.png, code-result-j06-02.png


## 読み取りどころ

- PSIが分布ずれの大きさを1数で表すこと（0.25超は大きな変化の目安）。申込時点で測れるので早期警報になること。
- 帯別の貸倒率が運用の健康診断になること。基準は学習に使っていないholdoutで測ること（学習に使った行では甘く出る）。
- 帯別の貸倒率は貸倒の確定を待たないと測れないこと。再学習は定期ではなく、確定後の条件で発火させること（条件自体が仮置きである点に注意）。